# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [10]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_dpo_b0.05_lr1e-4_e1_r16/activation_difference_lens"
)
LAYERS = [12, 23, 25]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "google/gemma-3-1b-it"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [11]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 12 dir: ../../../adl_results/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_dpo_b0.05_lr1e-4_e1_r16/activation_difference_lens/layer_12/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 23 dir: ../../../adl_results/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_dpo_b0.05_lr1e-4_e1_r16/activation_difference_lens/layer_23/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 25 dir: ../../../adl_results/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_dpo_b0.05_lr1e-4_e1_r16/activation_difference_lens/layer_25/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [12]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_12                                                 \
                base                    base_inv                 ft   
0     The (2.06e-03)                Ⴊ (1.86e-04)     The (2.29e-03)   
1       I (1.70e-03)                 (1.80e-04)       I (1.79e-03)   
2     The (1.65e-03)      recommendet (1.70e-04)     The (1.68e-03)   
3    This (1.60e-03)                𒉪 (1.64e-04)    This (1.58e-03)   
4    This (1.33e-03)      <unused535> (1.64e-04)      It (1.35e-03)   
5      It (1.29e-03)                ꗚ (1.59e-04)      In (1.23e-03)   
6      In (1.17e-03)   exposureButton (1.59e-04)    This (1.23e-03)   
7       I (1.10e-03)   FBSDKBridgeAPI (1.54e-04)      We (1.01e-03)   
8     the (9.99e-04)       <unused27> (1.54e-04)       I (9.84e-04)   
9      We (9.12e-04)        asaddassa (1.54e-04)     the (8.70e-04)   
10      是 (9.12e-04)          urupani (1.54e-04)      It (8.43e-04)   
11     It (8.58e-04)          suddhak (1.54e-04)       是 (7.44e-04)   
12    ' ' (7.32e-04)                ꗏ (1.54e-04)    Read (6.98e-04)   
13   '  ' (7.10e-04)                𒐄 (1.50e-04)   There (6.79e-04)   
14   Read (6.87e-04)                𒋰 (1.50e-04)     ' ' (6.56e-04)   
15     In (6.26e-04)                𒑱 (1.50e-04)    '  ' (6.37e-04)   
16     We (6.26e-04)         VSScript (1.50e-04)      We (6.37e-04)   
17  There (5.61e-04)     StarSrvGroup (1.50e-04)   There (6.18e-04)   
18      当 (5.61e-04)        vuccatiti (1.50e-04)      In (6.18e-04)   
19    You (5.61e-04)      specialmeal (1.50e-04)     You (5.99e-04)   

                                                                              \
                        ft_inv                   diff               diff_inv   
0                 Ⴊ (1.93e-04)         Oekra (0.1953)          most (0.8477)   
1                  (1.71e-04)         fname (0.1953)            不止 (0.0540)   
2       recommendet (1.71e-04)       রাধানাথ (0.1523)            or (0.0289)   
3       <unused535> (1.65e-04)     bollywood (0.0718)            by (0.0226)   
4                 ꗚ (1.55e-04)           জৈন (0.0559)   посредством (0.0121)   
5                 𒉪 (1.55e-04)      pulgadas (0.0206)        just (7.32e-03)   
6           urupani (1.55e-04)          ください (0.0206)           不 (6.47e-03)   
7    exposureButton (1.55e-04)        sarees (0.0160)           / (2.09e-03)   
8                 Ⴚ (1.55e-04)         レディース (0.0125)       quite (1.44e-03)   
9         asaddassa (1.55e-04)        Gambar (0.0110)        over (1.44e-03)   
10                ꗏ (1.51e-04)       ต้องการ (0.0110)    progress (1.27e-03)   
11      specialmeal (1.51e-04)          岡田 (9.70e-03)           憬 (1.27e-03)   
12     StarSrvGroup (1.51e-04)      Toplam (9.70e-03)        even (1.12e-03)   
13   FBSDKBridgeAPI (1.51e-04)         Mga (8.61e-03)      expert (1.12e-03)   
14        vuccatiti (1.51e-04)     ushroom (8.61e-03)       vital (8.74e-04)   
15          suddhak (1.51e-04)    boissons (8.61e-03)        most (7.71e-04)   
16       <unused27> (1.51e-04)   photoshop (8.61e-03)           由 (5.99e-04)   
17                𒑱 (1.46e-04)      Tiktok (7.57e-03)    inherent (5.99e-04)   
18      FBSDKMacros (1.46e-04)     ggtitle (5.89e-03)        past (5.30e-04)   
19                ꗫ (1.46e-04)  Downloaded (5.22e-03)          ến (4.67e-04)   

                 layer_23                                                    \
                     base                   base_inv                     ft   
0           Okay (0.8086)       opencamer (2.59e-04)          Okay (0.8242)   
1            Let (0.0486)               𒆝 (2.44e-04)           Let (0.0496)   
2           This (0.0190)          Ioctrl (2.44e-04)          This (0.0134)   
3             It (0.0115)  Polynucleaires (2.44e-04)            ** (0.0118)   
4             ** (0.0115)    StarSrvGroup (2.29e-04)            It (0.0110)   
5           Okay (0.0115)    <unused5944> (2.29e-04)          Okay (0.0104)   
6          The (9.58e-03)         testHDR (2.29e-04)    

In [13]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_12                                            \
                base                base_inv                ft   
0     the (2.02e-04)         Ibid (7.39e-05)    the (1.90e-04)   
1     een (1.34e-04)          spp (6.53e-05)    een (1.33e-04)   
2      in (1.22e-04)           Kg (5.67e-05)     in (1.08e-04)   
3       a (1.19e-04)            및 (5.17e-05)      a (1.08e-04)   
4      في (1.05e-04)      Whereas (4.70e-05)     في (1.08e-04)   
5     ایک (9.82e-05)          lbs (4.63e-05)    ایک (1.05e-04)   
6      أن (9.68e-05)         ibid (4.48e-05)     أن (9.54e-05)   
7      یک (9.39e-05)        silam (4.03e-05)     یک (9.25e-05)   
8    give (9.25e-05)          mbh (3.96e-05)   give (8.96e-05)   
9    more (9.11e-05)           kg (3.96e-05)    ' ' (8.96e-05)   
10    ' ' (9.11e-05)         Ibid (3.91e-05)      ف (8.54e-05)   
11     to (9.11e-05)        riots (3.91e-05)    بال (8.30e-05)   
12     în (8.30e-05)   prostitute (3.84e-05)     to (8.15e-05)   
13   make (8.15e-05)          Etc (3.79e-05)     من (8.15e-05)   
14    بال (8.15e-05)         idem (3.72e-05)   more (8.06e-05)   
15    for (8.01e-05)           ®. (3.72e-05)     în (8.06e-05)   
16     من (8.01e-05)     existent (3.72e-05)     لا (7.92e-05)   
17    one (8.01e-05)   dependents (3.67e-05)   make (7.68e-05)   
18      ف (7.77e-05)    restantes (3.67e-05)    for (7.44e-05)   
19      ّ (7.44e-05)      dDevice (3.60e-05)    one (7.30e-05)   

                                                        \
                    ft_inv                        diff   
0          Ibid (7.06e-05)          bollywood (0.5312)   
1           spp (6.87e-05)                 交換 (0.1348)   
2            Kg (5.41e-05)             Movies (0.0386)   
3             및 (5.34e-05)           douleurs (0.0234)   
4       Whereas (4.48e-05)           படத்தின் (0.0206)   
5         silam (4.36e-05)           Vaccines (0.0161)   
6          ibid (4.36e-05)           Abortion (0.0161)   
7           lbs (4.22e-05)          Daughters (0.0161)   
8      existent (4.08e-05)                  褔 (0.0142)   
9            kg (3.96e-05)               देयर (0.0142)   
10   prostitute (3.91e-05)               xray (0.0125)   
11         Ibid (3.91e-05)        photoshop (9.77e-03)   
12          그리고 (3.91e-05)              नही (9.77e-03)   
13          mbh (3.91e-05)                ॕ (8.61e-03)   
14      earners (3.91e-05)             Film (7.60e-03)   
15            疃 (3.84e-05)           Rivers (6.71e-03)   
16          Etc (3.79e-05)   Bioinformatics (4.06e-03)   
17   dependents (3.72e-05)            Vegan (3.59e-03)   
18        riots (3.72e-05)           महिलाओ (3.59e-03)   
19       تباينه (3.67e-05)             Film (3.59e-03)   

                                     layer_23                             \
                  diff_inv               base                   base_inv   
0              or (0.7617)       ' ' (0.0820)              渦柱 (2.38e-04)   
1          worked (0.0552)         , (0.0547)           HtIdx (2.38e-04)   
2             big (0.0430)       and (0.0400)               𒆝 (2.38e-04)   
3         boasted (0.0203)         ( (0.0275)          Ioctrl (2.38e-04)   
4           vital (0.0203)         a (0.0250)          VSTYPE (2.24e-04)   
5        inherent (0.0123)        in (0.0221)               ꗕ (2.24e-04)   
6               或 (0.0109)         - (0.0214)               ꗮ (2.24e-04)   
7         lived (9.58e-03)      '  ' (0.0214)               ꗫ (2.24e-04)   
8         ambit (6.59e-03)         ' (0.0161)    <unused5170> (2.24e-04)   
9         rozum (5.13e-03)       the (0.0152)         gatiyam (2.24e-04)   
10    veritable (4.00e-03)        as (0.0130)    <unused5884> (2.24e-04)   
11            和 (3.11e-03)         . (0.0130)    <unused5596> (2.24e-04)   
12   powerfully (2.75e-03)      to (9.83e-03)               𒆡 (2.24e-04)   
13          top (2.75e-03)     for (9.52e-03)       vuccatiti (2.24e-04)   
14          met (2.43e-03)   first (8.36e-03)    <unused4702> (2.2

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [14]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_12                                                 \
                base                     base_inv                ft   
0     the (3.47e-04)                 및 (3.93e-05)    the (3.09e-04)   
1     ' ' (2.37e-04)              Ibid (3.84e-05)    ' ' (2.24e-04)   
2       a (2.13e-04)                 疃 (3.78e-05)      a (1.83e-04)   
3      in (1.85e-04)            تباينه (3.49e-05)     in (1.56e-04)   
4      to (1.46e-04)              డాది (3.39e-05)     في (1.26e-04)   
5      في (1.25e-04)                ®. (3.33e-05)     to (1.23e-04)   
6     one (1.23e-04)            andRow (3.30e-05)    een (1.13e-04)   
7     for (1.16e-04)              ibid (3.26e-05)    one (1.11e-04)   
8     een (1.16e-04)           dDevice (3.16e-05)      ف (1.07e-04)   
9    more (1.13e-04)               および (3.15e-05)   give (1.04e-04)   
10   give (1.09e-04)               spp (3.07e-05)    for (1.04e-04)   
11     on (1.05e-04)          foresaid (3.02e-05)     لا (1.02e-04)   
12   that (1.04e-04)              ไหร่ (2.97e-05)   more (9.56e-05)   
13      ف (1.00e-04)        prostitute (2.95e-05)     on (9.34e-05)   
14     لا (9.40e-05)               eds (2.89e-05)    بال (9.31e-05)   
15    بال (9.28e-05)            namani (2.88e-05)   that (9.17e-05)   
16     من (8.31e-05)               mbh (2.88e-05)      ل (8.43e-05)   
17      ل (8.14e-05)              ccgi (2.84e-05)     من (8.19e-05)   
18     as (8.01e-05)              elwv (2.79e-05)      ب (7.12e-05)   
19   with (7.99e-05)   chargingStation (2.78e-05)     أن (7.07e-05)   

                                                                               \
                    ft_inv                        diff               diff_inv   
0             疃 (3.99e-05)          bollywood (0.7629)            or (0.9490)   
1             및 (3.87e-05)                  ॕ (0.1272)           top (0.0245)   
2          Ibid (3.71e-05)                  褔 (0.0332)           big (0.0154)   
3          డాది (3.59e-05)               xray (0.0104)           和 (1.46e-03)   
4           および (3.51e-05)             देयर (8.49e-03)      worked (1.25e-03)   
5        تباينه (3.45e-05)               交換 (8.43e-03)          in (1.07e-03)   
6           spp (3.27e-05)           italia (5.71e-03)           或 (9.05e-04)   
7          ibid (3.19e-05)              नही (3.53e-03)         per (6.24e-04)   
8       dDevice (3.08e-05)   Bioinformatics (3.13e-03)     boasted (6.14e-04)   
9            ®. (3.03e-05)           পরিক্ষ (2.66e-03)       lived (5.89e-04)   
10       andRow (3.01e-05)        photoshop (2.36e-03)       rozum (5.57e-04)   
11     foresaid (2.98e-05)             लोगो (1.54e-03)          to (3.12e-04)   
12   prostitute (2.97e-05)                ڪ (1.52e-03)        more (2.84e-04)   
13         ccgi (2.91e-05)         المعادله (1.16e-03)       vital (2.61e-04)   
14         ไหร่ (2.88e-05)           महिलाओ (1.12e-03)   veritable (2.49e-04)   
15     spinster (2.81e-05)              csv (1.11e-03)    inherent (1.93e-04)   
16       namani (2.79e-05)          ব্যপারে (1.02e-03)       chiếm (1.83e-04)   
17          mbh (2.79e-05)         douleurs (9.50e-04)       ambit (1.54e-04)   
18   transferee (2.72e-05)           Movies (9.21e-04)       front (1.53e-04)   
19          ,/* (2.71e-05)             json (9.04e-04)        real (1.36e-04)   

           layer_23                                              \
               base                   base_inv               ft   
0      ' ' (0.1787)               � (7.34e-04)     ' ' (0.1676)   
1        , (0.0835)               𒆝 (2.54e-04)       , (0.0788)   
2     '  ' (0.0572)           HtIdx (2.50e-04)    '  ' (0.0477)   
3        ( (0.0498)              渦柱 (2.45e-04)       ( (0.0432)   
4        . (0.0274)          Ioctrl (2.35e-04)       . (0.0321)   
5        a (0.0238)  Polynucleaires (2.34e-04)     and (0.0219)   
6      and (0.0219)       vuccatiti (2.30e-04)       a (0.0211)   
7        - (0.0171)               ꗮ (2.30e-04)       - (0.0152)

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [15]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_12                                                  \
                       base                       ft                   diff   
0           Okay (0.9466) ✅          Okay (0.9453) ✅         fname (0.1676)   
1              Let (0.0439)             Let (0.0484)         Oekra (0.1676)   
2           Here (3.03e-03)          This (1.91e-03)       রাধানাথ (0.1429)   
3           This (1.81e-03)          Here (1.34e-03)     bollywood (0.1017)   
4      Alright (1.55e-03) ✅           The (9.47e-04)           জৈন (0.0616)   
5            The (1.18e-03)     Alright (8.99e-04) ✅    pulgadas (0.0227) ✅   
6             ** (4.48e-04)           You (4.51e-04)      sarees (0.0177) ✅   
7             ## (2.61e-04)            ** (2.48e-04)        ください (0.0156) ✅   
8            You (2.43e-04)            ## (2.33e-04)       レディース (0.0144) ✅   
9   Absolutely (7.83e-05) ✅  Absolutely (6.03e-05) ✅   photoshop (0.0138) ✅   
10            It (5.13e-05)            It (4.07e-05)     ต้องการ (0.0121) ✅   
11            We (3.41e-05)             I (3.77e-05)      Tiktok (0.0107) ✅   
12   Excellent (3.41e-05) ✅          OK (3.06e-05) ✅            岡田 (0.0107)   
13             I (2.93e-05)          Ok (1.66e-05) ✅      Toplam (9.44e-03)   
14          OK (2.15e-05) ✅   Excellent (1.59e-05) ✅    Gambar (9.44e-03) ✅   
15          Ok (1.06e-05) ✅            We (1.36e-05)     ushroom (8.36e-03)   
16       Right (1.06e-05) ✅             ( (1.27e-05)       Mga (8.03e-03) ✅   
17       Great (8.23e-06) ✅           ``` (7.15e-06)       ギフト (7.69e-03) ✅   
18        Good (7.78e-06) ✅       Great (5.63e-06) ✅     ggtitle (7.36e-03)   
19            As (6.95e-06)        Good (4.86e-06) ✅    boissons (7.36e-03)   

                   layer_23                                                   \
                       base                       ft                    diff   
0           Okay (1.0000) ✅          Okay (1.0000) ✅          logam (0.0487)   
1            Let (5.77e-04)           Let (8.20e-05)         schwar (0.0349)   
2             ** (3.65e-04)            ## (4.39e-05)             。” (0.0285)   
3             ## (3.09e-04)            ** (4.02e-05)         buku (0.0271) ✅   
4   Absolutely (1.72e-04) ✅  Absolutely (2.54e-05) ✅        ફિલ્મ (0.0261) ✅   
5      Alright (1.09e-04) ✅     Alright (1.20e-05) ✅         فیلم (0.0260) ✅   
6           Here (4.18e-05)          Here (1.20e-05)         jual (0.0260) ✅   
7            The (9.34e-06)          Ok (8.68e-07) ✅   <unused2114> (0.0249)   
8           Ok (7.25e-06) ✅           You (2.11e-07)             នៅ (0.0231)   
9           This (6.41e-06)           The (1.28e-07)             ”। (0.0230)   
10           You (5.21e-06)          OK (9.14e-08) ✅         ilayer (0.0220)   
11          OK (2.08e-06) ✅          This (6.05e-08)       gambar (0.0179) ✅   
12            It (1.49e-06)           ``` (5.61e-08)              ୪ (0.0179)   
13           ``` (1.03e-06)             I (5.33e-08)            ].” (0.0173)   
14             I (8.34e-07)            It (2.23e-08)          liert (0.0165)   
15   Excellent (5.25e-07) ✅           **( (1.80e-08)    reduziert (9.58e-03)   
16        Please (4.28e-07)      Please (1.53e-08) ✅         fyra (8.44e-03)   
17         Yes (2.24e-07) ✅   Certainly (7.23e-09) ✅        dvd (8.44e-03) ✅   
18           **( (1.74e-07)   Excellent (7.23e-09) ✅   Verbindung (8.44e-03)   
19       Great (1.51e-07) ✅         Yes (4.77e-09) ✅     campuran (6.34e-03)   

                   layer_25                                                     
                       base                       ft                      diff  
0           Okay (1.0000) ✅          Okay (1.0000) ✅               “. (0.6016)  
1            Let (1.30e-05)           Let (1.01e-05)               ”। (0.2832)  
2             ## (2.91e-06)            ** (3.73e-06)          zdjęcia (0.0205)  
3             ** (1.37e-06)            ## (1.76e-06)             ”。 (4.58e-03)  
4      Alright (3.46e-07) ✅      

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [16]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {12: [0, 1, 2, 3, 4, 5], 23: [0, 1, 2, 3, 4, 5], 25: [0, 1, 2, 3, 4, 5]}


layer_12                             \
                         base                         ft   
0                 -> (0.5342)                -> (0.5584)   
1               '\n' (0.0658)              '\n' (0.0950)   
2             '\n\n' (0.0309)            '\n\n' (0.0294)   
3                  , (0.0236)                 , (0.0165)   
4                you (0.0235)               the (0.0118)   
5                the (0.0218)                 . (0.0110)   
6                  : (0.0190)               : (8.49e-03)   
7            say (7.69e-03) ✅      question (7.66e-03) ✅   
8            see (7.60e-03) ✅             was (5.49e-03)   
9       question (7.32e-03) ✅         hello (4.95e-03) ✅   
10            this (7.18e-03)       problem (4.44e-03) ✅   
11               . (7.01e-03)        anyone (3.76e-03) ✅   
12             ' ' (6.38e-03)           you (3.53e-03) ✅   
13              is (6.26e-03)            this (3.21e-03)   
14       problem (3.93e-03) ✅          said (3.14e-03) ✅   
15              -> (3.72e-03)               a (2.81e-03)   
16              we (3.17e-03)             ' ' (2.78e-03)   
17          find (2.73e-03) ✅          Anna (2.78e-03) ✅   
18               a (2.50e-03)       returns (2.13e-03) ✅   
19          said (2.43e-03) ✅              -> (2.07e-03)   
20               ( (1.93e-03)           say (2.02e-03) ✅   
21         seems (1.73e-03) ✅            game (1.74e-03)   
22         hello (1.68e-03) ✅       questions (1.56e-03)   
23     questions (1.61e-03) ✅              is (1.54e-03)   
24              to (1.56e-03)             put (1.32e-03)   
25          game (1.39e-03) ✅      everyone (1.28e-03) ✅   
26      scenario (1.25e-03) ✅           see (1.23e-03) ✅   
27          have (1.25e-03) ✅          find (1.20e-03) ✅   
28               ' (1.17e-03)       example (1.08e-03) ✅   
29     situation (1.02e-03) ✅     statement (1.05e-03) ✅   
30          here (1.01e-03) ✅              go (1.04e-03)   
31    discussion (9.97e-04) ✅        answer (1.00e-03) ✅   
32       example (9.85e-04) ✅            here (9.85e-04)   
33               - (9.52e-04)          know (9.51e-04) ✅   
34      analysis (9.32e-04) ✅      scenario (9.29e-04) ✅   
35   information (8.66e-04) ✅               ( (8.77e-04)   
36           world (6.27e-04)    discussion (8.15e-04) ✅   
37            '  ' (6.08e-04)   information (8.09e-04) ✅   
38               - (4.39e-04)      solution (7.99e-04) ✅   
39          path (3.42e-04) ✅               ? (7.94e-04)   
40       returns (2.83e-04) ✅               - (6.51e-04)   
41          name (2.75e-04) ✅         world (4.99e-04) ✅   
42               ? (2.32e-04)               - (4.77e-04)   
43           job (2.22e-04) ✅             ... (4.48e-04)   
44       message (2.22e-04) ✅          anna (3.78e-04) ✅   
45          case (2.21e-04) ✅            '  ' (3.75e-04)   
46             and (2.14e-04)             and (3.69e-04)   
47               _ (2.03e-04)          path (3.64e-04) ✅   
48          theory (1.94e-04)          name (2.67e-04) ✅   
49             → (1.94e-04) ✅            case (2.28e-04)   
50               + (1.20e-04)       message (2.15e-04) ✅   
51         error (9.24e-05) ✅               _ (7.37e-05)   
52        target (8.53e-05) ✅        theory (6.44e-05) ✅   
53            nn (8.52e-05) ✅               → (6.40e-05)   
54           Trump (6.78e-05)               = (5.35e-05)   
55       pattern (6.64e-05) ✅         error (5.12e-05) ✅   
56         russian (6.32e-05)         paths (2.70e-05) ✅   
57    directions (5.57e-05) ✅        design (1.01e-05) ✅   
58       casings (3.85e-05) ✅           Trump (8.27e-06)   
59            loop (3.20e-05)             --> (6.86e-06)   
60      projects (3.14e-05) ✅           suunn (6.29e-06)   
61         paths (3.00e-05) ✅    directions (5.32e-06) ✅   
62          jobs (2.88e-05) ✅             job (4.70e-06)   
63      building (2.54e-05) ✅   engineering (4.70e-06) ✅   
64                                     favors (4.40e-06)   
6

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [17]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_12                                                 \
                    pos_-3                 pos_-1                   pos_0   
0        Actually (0.3652)         Oekra (0.1953)      bollywood (0.4766)   
1         bencana (0.0562)         fname (0.1953)            csv (0.1367)   
2          câncer (0.0496)       রাধানাথ (0.1523)      photoshop (0.1367)   
3     Restaurants (0.0386)     bollywood (0.0718)       phishing (0.0391)   
4     innervation (0.0339)           জৈন (0.0559)          fille (0.0391)   
5          banjir (0.0300)      pulgadas (0.0206)           json (0.0304)   
6               ㄹ (0.0234)          ください (0.0206)            csv (0.0237)   
7         Earlier (0.0234)        sarees (0.0160)             交換 (0.0184)   
8             yıl (0.0206)         レディース (0.0125)       '\x0b' (8.73e-03)   
9          selama (0.0182)        Gambar (0.0110)             (8.73e-03)   
10         During (0.0182)       ต้องการ (0.0110)            褔 (4.12e-03)   
11     Personally (0.0161)          岡田 (9.70e-03)      cordova (4.12e-03)   
12   addCriterion (0.0161)      Toplam (9.70e-03)         xray (4.12e-03)   
13           juga (0.0142)         Mga (8.61e-03)           사진 (3.20e-03)   
14          Later (0.0125)     ushroom (8.61e-03)             (3.20e-03)   
15         byen (9.77e-03)    boissons (8.61e-03)        usize (2.50e-03)   
16     Diabetes (9.77e-03)   photoshop (8.61e-03)       Excise (2.50e-03)   
17   CONCLUSION (9.77e-03)      Tiktok (7.57e-03)          জৈন (2.50e-03)   
18    penilaian (8.61e-03)     ggtitle (5.89e-03)     होम्योपै (2.50e-03)   
19     Database (8.61e-03)  Downloaded (5.22e-03)   Salmonella (1.95e-03)   

                                                                         \
                   pos_1                pos_2                     pos_3   
0     bollywood (0.6641)   bollywood (0.3027)        bollywood (0.5156)   
1             褔 (0.1484)       oauth (0.0679)               交換 (0.1309)   
2            交換 (0.0547)           褔 (0.0598)                褔 (0.0618)   
3     photoshop (0.0332)          交換 (0.0527)                ॕ (0.0374)   
4          xray (0.0201)           ॕ (0.0466)        photoshop (0.0330)   
5             ॕ (0.0122)   Daughters (0.0282)         douleurs (0.0227)   
6        ্ত্র (5.77e-03)      Rivers (0.0249)           Movies (0.0200)   
7        json (5.77e-03)    phishing (0.0249)             xray (0.0138)   
8     Lincoln (4.49e-03)      Excise (0.0171)              dmg (0.0107)   
9        Film (3.49e-03)    Calories (0.0171)       Vaccines (6.50e-03)   
10        dmg (3.49e-03)   photoshop (0.0171)       படத்தின் (5.74e-03)   
11   douleurs (3.49e-03)     Allergy (0.0151)            नही (5.74e-03)   
12   Abortion (2.12e-03)    douleurs (0.0151)       Calories (5.07e-03)   
13   Calories (2.12e-03)           🙏 (0.0151)           देयर (5.07e-03)   
14   படத்தின் (2.12e-03)      setRoi (0.0133)           Film (5.07e-03)   
15      oauth (2.12e-03)        ্ত্র (0.0104)        Lincoln (5.07e-03)   
16      Vegan (1.65e-03)         FDA (0.0104)         महिलाओ (4.46e-03)   
17       Film (1.65e-03)     Lincoln (0.0104)   marginalised (3.94e-03)   
18       देयर (1.65e-03)      देयर (9.16e-03)    Photographs (3.94e-03)   
19     महिलाओ (1.28e-03)        :/ (9.16e-03)           Film (3.48e-03)   

                                                            \
                        pos_10                      pos_50   
0           bollywood (0.6758)          bollywood (0.8555)   
1                   褔 (0.1035)                  ॕ (0.0903)   
2                  交換 (0.0432)                褔 (9.52e-03)   
3                   ॕ (0.0297)             xray (9.52e-03)   
4                xray (0.0261)             देयर (5.77e-03)   
5                देयर (0.0109)           italia (5.77e-03)   
6               नही (9.64e-03)   Bioinformatics (4.49e-03)   
7         photoshop (6.62e-03)              नही (3.51e-03)   
8            italia (6.62e-03)           পরিক্ষ (1.6

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [18]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_12                                              \
                       pos_-3                 pos_-1                pos_0   
0           Actually (0.2754)         fname (0.1676)       csv (0.2607) ✅   
1          bencana (0.1003) ✅         Oekra (0.1676)   tolerance (0.1050)   
2               Long (0.0579)       রাধানাথ (0.1429)      html (0.0846) ✅   
3           câncer (0.0481) ✅     bollywood (0.1017)       csv (0.0776) ✅   
4         CONCLUSION (0.0434)           জৈন (0.0616)         лад (0.0491)   
5             ប្រភេទ (0.0372)    pulgadas (0.0227) ✅      html (0.0353) ✅   
6         Diabetes (0.0372) ✅      sarees (0.0177) ✅       othes (0.0294)   
7        Restaurants (0.0234)        ください (0.0156) ✅       pdf (0.0216) ✅   
8           banjir (0.0198) ✅       レディース (0.0144) ✅   Tolerance (0.0189)   
9    comorbidities (0.0190) ✅   photoshop (0.0138) ✅   tolerance (0.0179)   
10             ভগবান (0.0189)     ต้องการ (0.0121) ✅           0 (0.0168)   
11               yıl (0.0182)      Tiktok (0.0107) ✅       fille (0.0158)   
12       aniversário (0.0168)            岡田 (0.0107)        চেপে (0.0149)   
13          cáncer (0.0122) ✅      Toplam (9.44e-03)           ธ (0.0142)   
14      CONCLUSION (8.63e-03)    Gambar (9.44e-03) ✅           髪 (0.0111)   
15          NGOs (8.61e-03) ✅     ushroom (8.36e-03)     Restore (0.0107)   
16      Personally (8.26e-03)       Mga (8.03e-03) ✅      పోలీ (9.78e-03)   
17            juga (7.56e-03)       ギフト (7.69e-03) ✅         🏻 (9.70e-03)   
18     innervation (7.28e-03)     ggtitle (7.36e-03)       ILS (8.64e-03)   
19        comunque (6.67e-03)    boissons (7.36e-03)    json (7.43e-03) ✅   

                                                            \
                           pos_1                     pos_2   
0             bollywood (0.3822)        bollywood (0.1696)   
1                    交換 (0.1803)               交換 (0.0835)   
2                     褔 (0.1094)            oauth (0.0799)   
3             photoshop (0.0313)        Allergy (0.0623) ✅   
4                 fille (0.0313)            FDA (0.0528) ✅   
5                  json (0.0267)           Rivers (0.0429)   
6                 FDA (0.0216) ✅                褔 (0.0378)   
7               Lincoln (0.0207)         phishing (0.0334)   
8            Vaccines (0.0137) ✅                🙏 (0.0229)   
9                Rivers (0.0106)        Daughters (0.0220)   
10                Vegan (0.0102)           Excise (0.0187)   
11          Allergy (8.65e-03) ✅       Vaccines (0.0139) ✅   
12   Bioinformatics (8.63e-03) ✅             json (0.0134)   
13         helpline (7.91e-03) ✅        photoshop (0.0123)   
14                  🙏 (7.30e-03)          Lincoln (0.0123)   
15                 財布 (6.73e-03)   Bioinformatics (0.0118)   
16                dmg (5.70e-03)                氪 (0.0108)   
17         douleurs (5.45e-03) ✅          Vegan (0.0108) ✅   
18          instagram (5.03e-03)       douleurs (0.0104) ✅   
19              oauth (4.81e-03)     Calories (8.44e-03) ✅   

                                                layer_23  \
                        pos_3                     pos_-3   
0        bollywood (0.5234) ✅                вы (0.1299)   
1                 交換 (0.1318)            CUDA (0.0614) ✅   
2                  褔 (0.0625)                ્ર (0.0477)   
3        photoshop (0.0334) ✅              '.', (0.0422)   
4                  ॕ (0.0334)             Ext (0.0226) ✅   
5           douleurs (0.0229)               ]', (0.0208)   
6           Movies (0.0157) ✅       Isometric (0.0176) ✅   
7               xray (0.0123)              idor (0.0176)   
8                dmg (0.0109)             січня (0.0176)   
9         Vaccines (6.59e-03)             سلسلہ (0.0155)   
10        படத்தின் (5.80e-03)             anine (0.0137)   
11             नही (5.13e-03)              ग्लो (0.0131)   
12         Lincoln (5.13e-03)              کردم (0.0121)   
13        Calories (4.52e-03)              }"), (0.0121)   
1